# 🚦 Smart City Traffic Demand Prediction

---

## Objective

Develop a machine learning system capable of **accurately predicting traffic demand** (vehicles/hour) using historical traffic, road infrastructure, weather, and geolocation information.

## Models Used

| Model | Type | Key Strength |
|---|---|---|
| **Random Forest** | Bagging ensemble | Robust, low variance |
| **XGBoost** | Gradient boosting | High accuracy, regularisation |
| **LightGBM** | Leaf-wise gradient boosting | Speed + accuracy |
| **Weighted Ensemble** | Model averaging | Best overall generalisation |

## Evaluation Metrics

| Metric | Description |
|---|---|
| **R²** | Proportion of variance explained (higher = better) |
| **RMSE** | Root Mean Squared Error — penalises large errors |
| **MAE** | Mean Absolute Error — average magnitude of errors |
| **MAPE** | Mean Absolute Percentage Error — relative error (%) |

## Notebook Structure

1. Project Introduction  
2. Business Problem  
3. Dataset Description  
4. Import Libraries  
5. Load Dataset  
6. Exploratory Data Analysis  
7. Data Cleaning  
8. Feature Engineering  
9. Data Splitting  
10. Model Training  
11. Hyperparameter Tuning  
12. Ensemble Learning  
13. Model Evaluation  
14. Residual Analysis  
15. Feature Importance & SHAP  
16. Conclusion  
17. Future Improvements  

---
## 3. Import Libraries

In [ ]:
import os
import json
import joblib
import pandas as pd
os.chdir('..') # move to project root

# Load Test Data
test_df = pd.read_csv('data/processed/test.csv')
test_df['timestamp'] = pd.to_datetime(test_df['timestamp'])

# Load FeatureEngineer and transform test set
from src.features import FeatureEngineer, get_model_features_list
fe = FeatureEngineer.load('models/feature_engineer.pkl')
test_trans = fe.transform(test_df)
features = get_model_features_list()
target = 'traffic_demand'
X_test, y_test = test_trans[features], test_trans[target]

# Load Models
rf_model = joblib.load('models/rf_model.pkl')
xgb_model = joblib.load('models/xgb_model.pkl')
lgbm_model = joblib.load('models/lgbm_model.pkl')

# Load Ensemble Weights
with open('models/model_registry.json') as f:
    registry = json.load(f)
ens_weights = registry.get('ensemble_weights', {})
w_rf = ens_weights.get('rf', 0.0)
w_xgb = ens_weights.get('xgb', 0.45)
w_lgbm = ens_weights.get('lgbm', 0.55)

print("✅ Data, FeatureEngineer, Models, and Weights loaded successfully.")


In [ ]:
import os
import pandas as pd
os.chdir('..') # move to project root
test_df = pd.read_csv('data/processed/test.csv')
test_df['timestamp'] = pd.to_datetime(test_df['timestamp'])


---
## 12. Model Evaluation

We evaluate all models on the **held-out test set** (15% of data, chronologically last). This set was never seen during training, validation, or hyperparameter tuning.

In [ ]:
def mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

# Generate predictions
rf_preds   = rf_model.predict(X_test)
xgb_preds  = xgb_model.predict(X_test)
lgbm_preds = lgbm_model.predict(X_test)
ens_preds  = w_rf * rf_preds + w_xgb * xgb_preds + w_lgbm * lgbm_preds

# Compute metrics
models_eval = {
    'Random Forest' : rf_preds,
    'XGBoost'       : xgb_preds,
    'LightGBM'      : lgbm_preds,
    'Weighted Ensemble': ens_preds
}

results = []
for name, preds in models_eval.items():
    results.append({
        'Model' : name,
        'R²'    : r2_score(y_test, preds),
        'RMSE'  : np.sqrt(mean_squared_error(y_test, preds)),
        'MAE'   : mean_absolute_error(y_test, preds),
        'MAPE %': mape(y_test, preds)
    })

results_df = pd.DataFrame(results).set_index('Model')
print("📊 Test Set Performance Metrics:")
print(results_df.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# R² comparison bar chart
bar_colors = [PRIMARY, SECONDARY, ACCENT, RED]
r2_vals = results_df['R²']
bars = axes[0].bar(results_df.index, r2_vals, color=bar_colors, edgecolor='white', linewidth=1)
for bar, val in zip(bars, r2_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[0].set_title('R² Score by Model (Test Set)')
axes[0].set_ylabel('R² Score')
axes[0].tick_params(axis='x', rotation=15)
axes[0].set_ylim(min(r2_vals) * 1.2 if min(r2_vals) < 0 else 0, max(r2_vals) * 1.1)

# RMSE comparison
rmse_vals = results_df['RMSE']
bars2 = axes[1].bar(results_df.index, rmse_vals, color=bar_colors, edgecolor='white', linewidth=1)
for bar, val in zip(bars2, rmse_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[1].set_title('RMSE by Model (Test Set — lower is better)')
axes[1].set_ylabel('RMSE (vehicles/hour)')
axes[1].tick_params(axis='x', rotation=15)

plt.suptitle('Model Comparison — Test Set Metrics', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**Interpretation:**
- The **Weighted Ensemble** achieves the best R² and lowest RMSE on the held-out test set.
- **LightGBM** is the strongest individual model, followed closely by XGBoost — both benefit from sequential boosting.
- **Random Forest** provides useful diversity as a bagging estimator but scores lower alone.
- Since models were trained in **fast mode** (fixed params, no Optuna), running full Optuna HPO would further improve scores.

### 12.1 Actual vs. Predicted Plot

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.flatten()

sample_size = min(3000, len(y_test))
idx = np.random.choice(len(y_test), sample_size, replace=False)
y_s = np.array(y_test)[idx]

perfect_line = [y_test.min(), y_test.max()]

for ax, (name, preds), color in zip(axes, models_eval.items(), bar_colors):
    p_s = preds[idx]
    ax.scatter(y_s, p_s, alpha=0.25, s=12, color=color)
    ax.plot(perfect_line, perfect_line, 'k--', lw=2, label='Perfect Prediction')
    r2 = r2_score(y_s, p_s)
    ax.set_title(f'{name}\n(R² = {r2:.4f})')
    ax.set_xlabel('Actual Traffic Demand (veh/hr)')
    ax.set_ylabel('Predicted Traffic Demand (veh/hr)')
    ax.legend(fontsize=9)

plt.suptitle('Actual vs. Predicted — All Models (Test Set, n=3,000 sample)',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

**Ideal result:** Points tightly clustered along the 45° diagonal (`y = x`), indicating perfect predictions. Scatter around the diagonal represents error magnitude.

---
## 13. Residual Analysis

Residual analysis determines whether the model has **systematic bias** — structured patterns in its errors that suggest a missing feature or incorrect model form.

**Ideal residual plots:**
- **Histogram**: centred at 0, approximately normally distributed
- **Scatter (residual vs predicted)**: random cloud with no pattern, constant variance (homoscedasticity)

In [ ]:
residuals = np.array(y_test) - ens_preds

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# 1. Residual histogram
sns.histplot(residuals, bins=60, kde=True, color=PRIMARY, ax=axes[0])
axes[0].axvline(0, color=RED, linestyle='--', lw=2, label='Zero Error')
axes[0].axvline(residuals.mean(), color=SECONDARY, linestyle=':', lw=2, label=f'Mean={residuals.mean():.1f}')
axes[0].set_title('Residual Distribution — Ensemble')
axes[0].set_xlabel('Residual (Actual − Predicted)')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# 2. Residual vs Predicted scatter
axes[1].scatter(ens_preds[idx], residuals[idx], alpha=0.25, s=12, color=PRIMARY)
axes[1].axhline(0, color=RED, linestyle='--', lw=2)
axes[1].set_title('Residuals vs. Predicted Values')
axes[1].set_xlabel('Predicted Traffic Demand (veh/hr)')
axes[1].set_ylabel('Residual (Actual − Predicted)')

# 3. Residual vs Actual scatter
axes[2].scatter(np.array(y_test)[idx], residuals[idx], alpha=0.25, s=12, color=ACCENT)
axes[2].axhline(0, color=RED, linestyle='--', lw=2)
axes[2].set_title('Residuals vs. Actual Values')
axes[2].set_xlabel('Actual Traffic Demand (veh/hr)')
axes[2].set_ylabel('Residual (Actual − Predicted)')

plt.suptitle('Residual Analysis — Weighted Ensemble', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"  Residual Mean   : {residuals.mean():+.2f} (ideal: 0)")
print(f"  Residual Std    : {residuals.std():.2f}")
print(f"  Residual Skew   : {pd.Series(residuals).skew():+.4f}")
print(f"  % within ±500   : {(np.abs(residuals) <= 500).mean()*100:.1f}%")

### Error Analysis by Segment

In [ ]:
test_analysis = test_trans.copy()
test_analysis['ensemble_pred'] = ens_preds
test_analysis['residual']      = np.array(y_test) - ens_preds
test_analysis['abs_error']     = np.abs(test_analysis['residual'])

print("─" * 55)
print(f"{'Time Period':<20} {'Count':>7} {'Mean |Error|':>14} {'R²':>8}")
print("─" * 55)
for flag in ['early_morning','morning_rush','midday','evening_rush','night']:
    mask = test_analysis['hour_flag'] == flag
    sub  = test_analysis[mask]
    if len(sub) < 10: continue
    seg_r2 = r2_score(sub[target], sub['ensemble_pred'])
    print(f"{flag.replace('_',' ').title():<20} {mask.sum():>7,} {sub['abs_error'].mean():>14.1f} {seg_r2:>8.4f}")

print("─" * 55)
print()
print("─" * 55)
print(f"{'Road Type':<20} {'Count':>7} {'Mean |Error|':>14} {'R²':>8}")
print("─" * 55)
if 'road_type' in test_analysis.columns:
    for rt in sorted(test_analysis['road_type'].unique()):
        mask = test_analysis['road_type'] == rt
        sub  = test_analysis[mask]
        if len(sub) < 10: continue
        seg_r2 = r2_score(sub[target], sub['ensemble_pred'])
        print(f"{rt:<20} {mask.sum():>7,} {sub['abs_error'].mean():>14.1f} {seg_r2:>8.4f}")
print("─" * 55)

---
## 14. Feature Importance & SHAP Explainability

### 14.1 Model-based Feature Importance

Tree-based models provide **impurity-based importance** (mean decrease in impurity, or MDI). While fast, MDI can overestimate the importance of high-cardinality features.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 8))

for ax, (name, model), color in zip(
    axes,
    [('Random Forest', rf_model), ('XGBoost', xgb_model), ('LightGBM', lgbm_model)],
    [PRIMARY, SECONDARY, ACCENT]
):
    importances = model.feature_importances_
    feat_imp = pd.Series(importances, index=features).sort_values(ascending=True).tail(15)
    feat_imp.plot(kind='barh', ax=ax, color=color, edgecolor='white')
    ax.set_title(f'{name}\nTop-15 Feature Importances')
    ax.set_xlabel('Importance Score')
    ax.set_ylabel('')

plt.suptitle('Model-based Feature Importances (MDI)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### 14.2 SHAP Explainability

**SHAP (SHapley Additive exPlanations)** provides model-agnostic, theoretically grounded feature importance based on cooperative game theory.

**Why SHAP over MDI?**
- MDI is biased towards high-cardinality features
- SHAP values are **consistent** and **locally accurate** — they explain *individual predictions*, not just global averages
- SHAP can reveal both the magnitude and **direction** of each feature's effect

We use LightGBM (our best individual model) for SHAP analysis, as it has the fastest `TreeExplainer`.

In [ ]:
try:
    import shap
    shap.initjs()
    SHAP_AVAILABLE = True
    print(f"✅ SHAP {shap.__version__} available.")
except ImportError:
    SHAP_AVAILABLE = False
    print("⚠️  SHAP not installed. Run: uv pip install shap")
    print("   Re-run this cell after installation to see SHAP plots.")

In [ ]:
if SHAP_AVAILABLE:
    # Sample test set for SHAP (full can be slow)
    shap_sample_size = min(1000, len(X_test))
    X_shap = X_test.sample(shap_sample_size, random_state=42)

    print(f"Computing SHAP values for {shap_sample_size} test samples via LightGBM TreeExplainer...")
    explainer   = shap.TreeExplainer(lgbm_model)
    shap_values = explainer.shap_values(X_shap)

    # ── SHAP Summary (Beeswarm) ──
    print("\n📊 SHAP Beeswarm Plot — Impact of each feature on model output")
    plt.figure(figsize=(10, 8))
    shap.summary_plot(shap_values, X_shap, plot_type='dot', show=False, max_display=20)
    plt.title('SHAP Beeswarm Plot — LightGBM (Top-20 Features)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # ── SHAP Bar Plot (Global Importance) ──
    print("\n📊 SHAP Bar Plot — Mean |SHAP| per feature")
    plt.figure(figsize=(10, 8))
    shap.summary_plot(shap_values, X_shap, plot_type='bar', show=False, max_display=20)
    plt.title('SHAP Global Feature Importance — LightGBM', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # ── Top feature SHAP dependence plot ──
    shap_abs_mean = np.abs(shap_values).mean(axis=0)
    top_feature   = X_shap.columns[np.argmax(shap_abs_mean)]
    second_feature = X_shap.columns[np.argsort(shap_abs_mean)[-2]]

    print(f"\n📊 SHAP Dependence Plot — '{top_feature}' (coloured by '{second_feature}')")
    plt.figure(figsize=(10, 5))
    shap.dependence_plot(
        top_feature, shap_values, X_shap,
        interaction_index=second_feature,
        show=False, alpha=0.5
    )
    plt.title(f'SHAP Dependence: {top_feature}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # Print global ranking
    shap_importance = pd.Series(
        shap_abs_mean, index=X_shap.columns
    ).sort_values(ascending=False)
    print("\n🏆 SHAP Global Feature Ranking (LightGBM):")
    for i, (feat, val) in enumerate(shap_importance.head(15).items(), 1):
        print(f"  {i:>2}. {feat:<35} SHAP = {val:.2f}")
else:
    print("SHAP not available — skipping explainability plots.")
    print("Install with: uv pip install shap && restart kernel")

**SHAP Interpretation Guide:**
- **Beeswarm plot:** Each dot = one prediction. Colour = feature value (red=high, blue=low). X-axis = SHAP value (positive pushes prediction up, negative pushes it down).
- **Bar plot:** Mean absolute SHAP value — the global average importance.
- **Dependence plot:** Shows how a feature's SHAP value changes across its range, coloured by a correlated feature to reveal interactions.

**Key drivers of traffic demand (expected):**
- `hour_location_encoded` / `hour_sin` / `hour_cos` — time of day drives most demand
- `large_vehicles_count` — direct congestion signal
- `road_type_encoded` — highway vs residential capacity
- `traffic_density_score` — compound congestion feature
- `rush_hour_indicator` — binary peak-period flag

---
## 15. Conclusion

### Summary of Results

We trained a three-model weighted ensemble to predict urban traffic demand from historical sensor data. The pipeline covers end-to-end data science: loading, cleaning, feature engineering, training, tuning, and explainability.

In [ ]:
# Final summary table
summary_df = results_df.copy()
summary_df = summary_df.style \
    .format({'R²': '{:.4f}', 'RMSE': '{:.2f}', 'MAE': '{:.2f}', 'MAPE %': '{:.2f}%'}) \
    .highlight_max(subset=['R²'],    color='#d4edda') \
    .highlight_min(subset=['RMSE', 'MAE', 'MAPE %'], color='#d4edda') \
    .set_table_styles([{
        'selector': 'th',
        'props': [('background-color', '#2c3e50'), ('color', 'white'), ('font-weight', 'bold')]
    }])
summary_df

### Key Findings

| Finding | Detail |
|---|---|
| **Best model** | Weighted Ensemble (LightGBM + XGBoost dominant) |
| **Strongest predictors** | `hour_location_encoded`, `large_vehicles_count`, `road_type_encoded`, `traffic_density_score`, `rush_hour_indicator` |
| **Critical insight** | Temporal and spatial target encodings dramatically boost accuracy by capturing local, time-of-day demand patterns |
| **Cyclical encoding** | Essential — raw `hour` would misinform the model about midnight adjacency |
| **Ensemble benefit** | Combining diverse models reduces variance and improves generalisation over any single model |
| **Temporal split** | Honest evaluation — test metrics reflect true future generalisation, not random-split inflation |

### Model Suitability for Deployment

The Weighted Ensemble is suitable for deployment in a Smart City Traffic Management System for:

- ✅ **Real-time signal control** (fast inference via LightGBM/XGBoost)
- ✅ **Batch demand forecasting** (hourly predictions for the next 24 hours)
- ✅ **Route recommendation engines** (segment-level demand estimates)

> ⚠️ **Important:** Running the training pipeline in **Full Optuna HPO mode** (`train.py full`) will trigger 60–80 trial Bayesian optimization per model, which would substantially improve the R² metric. The current models used fast-mode fixed hyperparameters for rapid development.

---
## 16. Future Improvements

### Near-term

| Improvement | Expected Impact |
|---|---|
| **Full Optuna HPO** (80+ trials per model) | +2–5% R² improvement |
| **Holiday calendar integration** | Better handling of public holiday anomalies |
| **Lag features** (demand at t-1, t-7days) | Capture autocorrelation |
| **Expanded weather data** (UV index, visibility) | Richer weather signals |
| **Accident/event feeds** | Incorporate real-time disruption signals |

### Medium-term

| Improvement | Detail |
|---|---|
| **GPS trajectory data** | True origin-destination flows beyond sensor counts |
| **Graph Neural Networks** | Model road network topology explicitly |
| **LSTM / Transformer** | Deep sequence models for multi-step forecasting |
| **Stacking ensemble** | Use meta-learner instead of fixed weights |

### Deployment

| Step | Detail |
|---|---|
| **Live traffic API integration** | Ingest real-time sensor data |
| **Real-time inference endpoint** | FastAPI or TorchServe serving layer |
| **Drift detection** | Monitor prediction–actuals divergence over time |
| **Automated retraining** | Trigger retraining when drift threshold is exceeded |
| **Digital twin integration** | Feed predictions into city simulation models |

---

## Notebook Summary

```
┌───────────────────────────────────────────────────────────────┐
│            Smart City Traffic Demand Prediction                │
│                     Notebook Pipeline                         │
│                                                               │
│  Raw CSV → Clean → Engineer → Split → Train → Tune → Ensemble │
│         → Evaluate → Residuals → SHAP → Deploy                │
│                                                               │
│  Best Model: Weighted Ensemble (LightGBM 55% + XGBoost 45%)  │
│  Feature Set: 34 features (temporal, spatial, weather)        │
└───────────────────────────────────────────────────────────────┘
```

*Developed as part of the UrbanFlow Smart City Traffic Intelligence Project.*